# A/B Testing & Causal Inference

Running valid experiments, avoiding common traps (peeking, SUTVA violations), and measuring causal effects rather than correlations.

## The peeking problem — why you can't check results early

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

# Simulate 1000 A/A tests (no true effect) where analyst peeks daily
n_days     = 30
n_per_day  = 50
n_sims     = 1000
alpha      = 0.05

fp_fixed_horizon = 0  # check only at end
fp_peeking       = 0  # check every day, stop at first significance

for _ in range(n_sims):
    control   = []
    treatment = []
    found_early = False
    for day in range(n_days):
        control   += rng.binomial(1, 0.10, n_per_day).tolist()
        treatment += rng.binomial(1, 0.10, n_per_day).tolist()
        # Peeking: test every day
        if len(control) >= 20:
            _, p = stats.ttest_ind(control, treatment)
            if p < alpha and not found_early:
                fp_peeking += 1
                found_early = True
    # Fixed horizon: test only at end
    _, p = stats.ttest_ind(control, treatment)
    if p < alpha:
        fp_fixed_horizon += 1

print(f"False positive rate (fixed horizon, α=0.05): {fp_fixed_horizon/n_sims:.1%}  (expected ≈5%)")
print(f"False positive rate (peek every day):         {fp_peeking/n_sims:.1%}  (inflated!)")
print()
print("Fix: commit to sample size before starting, or use sequential testing (CUPED/mSPRT)")

## CUPED — variance reduction with pre-experiment data

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(42)
n   = 1000

# Pre-experiment metric (highly correlated with post-experiment)
pre  = rng.normal(100, 20, n)
# Post-experiment: treatment lifts mean by 3 units
treat = rng.binomial(1, 0.5, n)
post  = pre * 0.8 + treat * 3 + rng.normal(0, 15, n)

# Standard t-test
t_std, p_std = stats.ttest_ind(post[treat==1], post[treat==0])

# CUPED adjustment: regress pre out
theta = np.cov(post, pre)[0,1] / np.var(pre)
post_cuped = post - theta * (pre - pre.mean())
t_cup, p_cup = stats.ttest_ind(post_cuped[treat==1], post_cuped[treat==0])

var_reduction = 1 - np.var(post_cuped) / np.var(post)
print(f"True ATE:            3.0")
print(f"Naive estimate:      {post[treat==1].mean() - post[treat==0].mean():.2f}")
print(f"CUPED estimate:      {post_cuped[treat==1].mean() - post_cuped[treat==0].mean():.2f}")
print()
print(f"Standard p-value:    {p_std:.4f}")
print(f"CUPED p-value:       {p_cup:.4f}")
print(f"Variance reduction:  {var_reduction:.1%}")
print("CUPED needs fewer samples to achieve the same power")

## Causal forest — heterogeneous treatment effects

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_predict

rng = np.random.default_rng(42)
n   = 3000

# Covariates: age, spend level
age   = rng.uniform(20, 70, n)
spend = rng.lognormal(4, 1, n)

# Treatment assignment (not random — older users more likely treated)
p_treat = 1 / (1 + np.exp(-(age - 45) / 10))
treat   = rng.binomial(1, p_treat, n)

# Heterogeneous effect: young high-spenders respond more
true_cate = 5 + 0.1 * (60 - age) + 0.001 * spend
outcome   = 20 + 0.05 * age + 0.005 * spend + true_cate * treat + rng.normal(0, 5, n)

X = np.column_stack([age, spend])

# Two-model (T-learner) CATE estimate
gbm1 = GradientBoostingRegressor(n_estimators=100, random_state=42)
gbm0 = GradientBoostingRegressor(n_estimators=100, random_state=42)
gbm1.fit(X[treat==1], outcome[treat==1])
gbm0.fit(X[treat==0], outcome[treat==0])
cate_hat = gbm1.predict(X) - gbm0.predict(X)

corr = np.corrcoef(true_cate, cate_hat)[0,1]
print(f"Correlation of true CATE vs estimated CATE: {corr:.3f}")
print(f"True mean CATE:      {true_cate.mean():.2f}")
print(f"Estimated mean CATE: {cate_hat.mean():.2f}")
print()
print("High-CATE users (top quartile):")
top = cate_hat > np.percentile(cate_hat, 75)
print(f"  Mean estimated effect: {cate_hat[top].mean():.2f}")
print(f"  Mean true effect:      {true_cate[top].mean():.2f}")
print("Target the high-CATE segment for maximum ROI")